## 1. Tokenizing text

In [1]:
with open("alice-ch01.txt", "r") as f:
    raw_text = f.read()
print(f"Length of text: {len(raw_text)} characters")

Length of text: 11274 characters


In [3]:
import re

text = "Hello, world! This is a test. Are you ready?"
result = re.split(r'(\s|[.,:;?_!"()\']|--)', text)
result = [token for token in result if token.strip()]
print(result)

['Hello', ',', 'world', '!', 'This', 'is', 'a', 'test', '.', 'Are', 'you', 'ready', '?']


In [4]:
preprocessed = re.split(r'(\s|[.,:;?_!"()\']|--|—)', raw_text)
preprocessed = [token for token in preprocessed if token.strip()]
print(f"Number of tokens: {len(preprocessed)}")

Number of tokens: 2622


## 2. Converting tokens into token IDs
Converting tokens into token IDs is a crucial step in preparing text data for language models.
To convert tokens into token IDs, we typically create a vocabulary that maps each unique token to a unique integer ID. This process is often referred to as "tokenization" and "encoding".

In [5]:
# Create a vocabulary of unique tokens, sorted alphabetically
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 646


In [6]:
# Create a mapping from each token to a unique integer ID
vocab = {word: idx for idx, word in enumerate(all_words)}

In [12]:
class SimpleTokenizer:
    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab
        self.inverse_vocab = {idx: word for word, idx in vocab.items()}

    def encode(self, text: str) -> list[int]:
        tokens = re.split(r'(\s|[.,:;?_!"()\']|--|—)', text)
        tokens = [token for token in tokens if token.strip()]
        token_ids = [self.vocab[token] for token in tokens]
        return token_ids

    def decode(self, token_ids: list[int]) -> str:
        tokens = [self.inverse_vocab[token_id] for token_id in token_ids]
        text = ' '.join(tokens)
        # Remove spaces before punctuation
        text = re.sub(r'\s+([.,:;?!"()\'])', r'\1', text)
        return text

In [13]:
tokenizer = SimpleTokenizer(vocab)
text = '"What a curious feeling!" said Alice; "I must be shutting up like a telescope."'
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 52, 58, 160, 221, 0, 1, 476, 11, 8, 1, 28, 388, 96, 503, 596, 341, 58, 548, 6, 1]
" What a curious feeling!" said Alice;" I must be shutting up like a telescope."


## 3. Adding special context tokens

Add some special tokens for unknown tokens, end of sequence, padding, etc.

Some common special tokens include:
- **[BOS]** (Beginning of Sequence): indicate the start of a sequence.
- **[EOS]** (End of Sequence): indicate the end of a sequence. `<|endoftext|>` is analogous to `[EOS]`. GPT-2 only uses `<|endoftext|>` as a special token to indicate the end of a sequence.
- **[PAD]**: used to pad sequences to a fixed length, ensuring that all input sequences have the same length for batch processing.
- **[UNK]**: used to represent unknown or out-of-vocabulary tokens that are not present in the vocabulary.

In [14]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token: idx for idx, token in enumerate(all_tokens)}
print(f"Vocabulary size with special tokens: {len(vocab)}")

Vocabulary size with special tokens: 648


In [15]:
for token in list(vocab.items())[-3:]:
    print(f"{token}")

('—', 645)
('<|endoftext|>', 646)
('<|unk|>', 647)


In [16]:
class SimpleTokenizerV2:
    """A simple tokenizer that converts text to token IDs and back, with support for unknown tokens."""
    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab
        self.inverse_vocab = {idx: word for word, idx in vocab.items()}

    def encode(self, text: str) -> list[int]:
        tokens = re.split(r'(\s|[.,:;?_!"()\']|--|—)', text)
        tokens = [token for token in tokens if token.strip()]
        token_ids = [self.vocab.get(token, self.vocab["<|unk|>"]) for token in tokens]
        return token_ids

    def decode(self, token_ids: list[int]) -> str:
        tokens = [self.inverse_vocab[token_id] for token_id in token_ids]
        text = ' '.join(tokens)
        # Remove spaces before punctuation
        text = re.sub(r'\s+([.,:;?!"()\'])', r'\1', text)
        return text

In [17]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "now I'm opening out like the largest telescope that ever was!"
text2 = "Good-bye, feet!"
text = " <|endoftext|> ".join((text1, text2))

print(text)

now I'm opening out like the largest telescope that ever was! <|endoftext|> Good-bye, feet!


In [18]:
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

[403, 28, 2, 362, 647, 417, 341, 554, 647, 548, 553, 208, 610, 0, 646, 647, 5, 222, 0]
now I' m <|unk|> out like the <|unk|> telescope that ever was! <|endoftext|> <|unk|>, feet!


## 4. BPE (Byte Pair Encoding)

BPE is a subword tokenization technique that breaks down words into smaller units, allowing the model to handle out-of-vocabulary words more effectively. It works by iteratively merging the most frequent pairs of characters or character sequences in the text until a predefined vocabulary size is reached.

GPT-2 used BPE as its tokenizer.

In [35]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
text = "now I'm opening out like the largest telescope that ever was! <|endoftext|> Good-bye, feet!"
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
print(tokenizer.decode(integers))

[2197, 314, 1101, 4756, 503, 588, 262, 4387, 24344, 326, 1683, 373, 0, 220, 50256, 4599, 12, 16390, 11, 3625, 0]
now I'm opening out like the largest telescope that ever was! <|endoftext|> Good-bye, feet!


## 5. Data sampling with a Sliding window


In [38]:
with open("alice-ch01.txt", "r") as f:
    raw_text = f.read()

tokens = tokenizer.encode(raw_text)
print(f"Number of tokens: {len(tokens)}")

Number of tokens: 2702


In [39]:
sample = tokens[50:]

context_size = 4
x = sample[:context_size]
y = sample[1:context_size + 1]
print(f"Input tokens (x): {x}")
print(f"Target tokens (y):     {y}")

Input tokens (x): [392, 644, 318, 262]
Target tokens (y):     [644, 318, 262, 779]


In [40]:
for i in range(1, context_size + 1):
    context = sample[:i]
    desired = sample[i]
    print(f"Context: {context} -> Desired next token: {desired}")

Context: [392] -> Desired next token: 644
Context: [392, 644] -> Desired next token: 318
Context: [392, 644, 318] -> Desired next token: 262
Context: [392, 644, 318, 262] -> Desired next token: 779


In [41]:
from typing import Any

import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt: str, tokenizer: Any, context_size: int, stride: int):
        """A PyTorch Dataset that generates input-target pairs.
        Args:
            txt (str): The input text to tokenize and create pairs from.
            tokenizer (Any): A tokenizer object with an encode method.
            context_size (int): The number of tokens in the input sequence.
            stride (int): The step size for moving the context window.
        """
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - context_size, stride):
            input_chunks = token_ids[i : i + context_size]
            target_chunks = token_ids[i + 1 : i + context_size + 1]
            self.input_ids.append(torch.tensor(input_chunks))
            self.target_ids.append(torch.tensor(target_chunks))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(
    txt,
    context_size=256,
    stride=128,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDatasetV1(txt, tokenizer, context_size, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [42]:
dataloader = create_dataloader_v1(
    raw_text, context_size=4, stride=1, batch_size=1, shuffle=False
)
data_iter = iter(dataloader)
for _ in range(3):
    x, y = next(data_iter)
    print(f"Input tokens (x): {x.tolist()[0]}")
    print(f"Target tokens (y): {y.tolist()[0]}")

Input tokens (x): [44484, 373, 3726, 284]
Target tokens (y): [373, 3726, 284, 651]
Input tokens (x): [373, 3726, 284, 651]
Target tokens (y): [3726, 284, 651, 845]
Input tokens (x): [3726, 284, 651, 845]
Target tokens (y): [284, 651, 845, 10032]


In [43]:
# Make stride equal to context_size to avoid overlapping between batches
# This helps model better learn relationships between different contexts and reduces the risk of overfitting.

dataloader = create_dataloader_v1(
    raw_text, context_size=4, stride=4, batch_size=4, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Input tokens (x): {inputs.tolist()}")
print(f"Target tokens (y): {targets.tolist()}")

Input tokens (x): [[44484, 373, 3726, 284], [651, 845, 10032, 286], [5586, 416, 607, 6621], [319, 262, 3331, 11]]
Target tokens (y): [[373, 3726, 284, 651], [845, 10032, 286, 5586], [416, 607, 6621, 319], [262, 3331, 11, 290]]


## 6. Creating token embeddings

Embedding layers are used to convert token IDs into vector representations that can be processed by the model.

In [44]:
input_ids = torch.tensor([1, 3, 4, 5])

vocab_size = 6
out_dim = 3
torch.manual_seed(42)
embedding_layer = torch.nn.Embedding(vocab_size, out_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 1.9269,  1.4873, -0.4974],
        [ 0.4396, -0.7581,  1.0783],
        [ 0.8008,  1.6806,  0.3559],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]], requires_grad=True)


In [45]:
print(embedding_layer(input_ids))

tensor([[ 0.4396, -0.7581,  1.0783],
        [-0.6866,  0.6105,  1.3347],
        [-0.2316,  0.0418, -0.2516],
        [ 0.8599, -0.3097, -0.3957]], grad_fn=<EmbeddingBackward0>)


## 7. Encoding word positions

LLM's self-attention mechanism is unordered, requiring positional encoding to capture the order of tokens in a sequence. There are two common methods for positional encoding:
- **Absolute positional encoding**: Assigns a unique absolute position ID to each token in the input sequence.
- **Relative positional encoding**: Encodes positional information based on the relative positions between tokens in the input sequence. Relative positional encoding is better at capturing long-range dependencies.

In [46]:
vocab_size = 50257  # GPT-2's vocabulary size
out_dim = 256
token_embedding_layer = torch.nn.Embedding(vocab_size, out_dim)

In [47]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, context_size=max_length, stride=max_length, batch_size=8, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(f"Input tokens: {inputs.tolist()}")
print(f"Input shape: {inputs.shape}")

Input tokens: [[44484, 373, 3726, 284], [651, 845, 10032, 286], [5586, 416, 607, 6621], [319, 262, 3331, 11], [290, 286, 1719, 2147], [284, 466, 25, 1752], [393, 5403, 673, 550], [613, 538, 276, 656]]
Input shape: torch.Size([8, 4])


In [48]:
token_embeddings = token_embedding_layer(inputs)
print(f"Token embeddings shape: {token_embeddings.shape}")

Token embeddings shape: torch.Size([8, 4, 256])


In [49]:
# create position embeddings
pos_embedding_layer = torch.nn.Embedding(max_length, out_dim)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(f"Position embeddings shape: {pos_embeddings.shape}")

Position embeddings shape: torch.Size([4, 256])


In [50]:
input_embeddings = token_embeddings + pos_embeddings
print(f"Input embeddings shape: {input_embeddings.shape}")

Input embeddings shape: torch.Size([8, 4, 256])
